# Semana 39 · ¿Cuántas estrellas hay cerca del Sol y de qué tipo?

Este cuaderno acompaña a la sesión de proyecto de la semana 39: estadística descriptiva con las estrellas del vecindario solar, completitud de un catálogo y sesgos de selección.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Los datos y el tipo espectral

Cargamos el catálogo HYG y clasificamos cada estrella por la letra de su tipo espectral. Los tipos vienen escritos de muchas formas (`M2V`, `dM4e`, `k-m`, `DA2` para las enanas blancas...), así que usamos una **expresión regular**, una pequeña fórmula para buscar patrones en un texto, con el módulo `re` de Python.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/alabcar/hola-universo/main/cuadernos/datos/estrellas_hyg.csv"
locales = ["cuadernos/datos/estrellas_hyg.csv", "../datos/estrellas_hyg.csv", "datos/estrellas_hyg.csv"]
ruta = next((r for r in locales if os.path.exists(r)), URL)
estrellas = pd.read_csv(ruta)
print(f"{len(estrellas)} estrellas leídas de {ruta}")

In [ ]:
import re

def clase(espectro):
    if not isinstance(espectro, str) or not espectro.strip():
        return "sin clasificar"
    e = espectro.strip()
    if re.match(r"^D[ABCOQZXP]", e):              # DA, DB, DC... son enanas blancas
        return "enana blanca"
    if e[0] in "dD":                               # el prefijo d significa «enana» (dwarf)
        e = e[1:]
    m = re.search(r"[OBAFGKM]", e.upper())
    return m.group(0) if m else "sin clasificar"

est = estrellas[estrellas.dist_pc > 0].copy()
est["clase"] = est.espectro.map(clase)
cerca = est[est.dist_pc < 10]

orden = ["O", "B", "A", "F", "G", "K", "M", "enana blanca", "sin clasificar"]
tabla = cerca.clase.value_counts().reindex(orden, fill_value=0)
tabla = pd.DataFrame({"número": tabla, "porcentaje": (100 * tabla / tabla.sum()).round(1)})
print(f"Estrellas a menos de 10 pc en el catálogo: {len(cerca)}")
tabla

In [ ]:
colores = {"A": "#cad7ff", "F": "#f8f7ff", "G": "#fff4ea", "K": "#ffd2a1", "M": "#ffad51", "enana blanca": "#9db4ff", "sin clasificar": "gray"}
t = tabla[tabla["número"] > 0]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(t.index, t["número"], color=[colores.get(k, "gray") for k in t.index], edgecolor="black")
ax.set_ylabel("número de estrellas"); ax.set_title("Estrellas a menos de 10 pc por tipo espectral")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 2. Histogramas y medidas de centro y dispersión

La magnitud absoluta de las estrellas a menos de 10 pc, con tres anchuras de intervalo distintas.

In [ ]:
M = cerca.absmag
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, ancho in zip(axs, [4, 1, 0.25]):
    bordes = np.arange(np.floor(M.min()), np.ceil(M.max()) + ancho, ancho)
    ax.hist(M, bins=bordes, color="tab:blue", edgecolor="white")
    ax.axvline(4.83, color="orange", lw=2, label="el Sol")
    ax.set_title(f"intervalos de {ancho} magnitudes"); ax.set_xlabel("magnitud absoluta M"); ax.legend()
plt.tight_layout(); plt.show()

print(f"media:     {M.mean():.2f}")
print(f"mediana:   {M.median():.2f}")
print(f"desv. típica: {M.std():.2f}")
print(f"cuartiles: {M.quantile([0.25, 0.5, 0.75]).round(2).tolist()}")
print(f"más luminosas que el Sol: {(M < 4.83).mean() * 100:.0f} %")

Con intervalos de 4 magnitudes se pierde la forma; con 0,25, el histograma se llena de huecos al azar. Con 1 magnitud se ve bien que la mayoría de las estrellas cercanas tiene magnitudes absolutas entre 9 y 15: enanas rojas.

**Ejercicio.** Calcula la media y la mediana de la **luminosidad** (`lum_sol`) de las mismas estrellas. ¿Por qué salen tan distintas?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
L = cerca.lum_sol
print(f"media de la luminosidad: {L.mean():.2f} L☉   mediana: {L.median():.4f} L☉")
print("Unas pocas estrellas muy luminosas (Sirio, Vega...) arrastran la media; la mediana es una enana roja.")

## 3. ¿Está completo el catálogo?

Si las estrellas se repartieran con densidad uniforme y el catálogo estuviera completo, el número de estrellas dentro de un radio $r$ crecería como el volumen: $N(<r) \propto r^3$. Lo comparamos con lo que hay.

In [ ]:
r = np.linspace(2, 25, 100)
N = np.array([(est.dist_pc < ri).sum() for ri in r])
densidad_5pc = (est.dist_pc < 5).sum() / (4 / 3 * np.pi * 5**3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.plot(r, N, label="catálogo HYG")
a1.plot(r, densidad_5pc * 4 / 3 * np.pi * r**3, "k--", label="r³ con la densidad de los primeros 5 pc")
a1.set_xlabel("radio (pc)"); a1.set_ylabel("estrellas dentro del radio"); a1.legend(); a1.grid(alpha=0.3)
for limite, color in [(9, "tab:red"), (12, "tab:green"), (16, "tab:purple")]:
    sub = est[est.absmag < limite]
    Nl = np.array([(sub.dist_pc < ri).sum() for ri in r])
    a2.plot(r, Nl / (4 / 3 * np.pi * r**3), color=color, label=f"M < {limite}")
a2.set_xlabel("radio (pc)"); a2.set_ylabel("densidad (estrellas por pc³)"); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"densidad en los primeros 5 pc: {densidad_5pc:.3f} estrellas por pc³")

A partir de unos 6-8 pc, el catálogo se queda por debajo de la curva $r^3$: faltan estrellas. A la derecha se ve por qué: las estrellas luminosas ($M < 9$) tienen una densidad casi constante desde unos 8 pc hasta 25 pc (están casi todas; por debajo de 8 pc hay tan pocas que la densidad fluctúa mucho), pero las más débiles ($M < 16$ incluye las enanas rojas) se van perdiendo con la distancia, porque su magnitud aparente pasa del límite de los catálogos.

## 4. Limitado en volumen frente a limitado en brillo

Comparamos las estrellas a menos de 10 pc con las más brillantes que la magnitud aparente 4: dos «censos» que cuentan historias muy distintas.

In [ ]:
brillantes = est[est.mag < 4]
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
for ax, datos, titulo in [(axs[0], cerca, "limitado en volumen (d < 10 pc)"), (axs[1], brillantes, "limitado en brillo (m < 4)")]:
    t = datos.clase.value_counts().reindex(orden, fill_value=0)
    t = t[t > 0]
    ax.bar(t.index, 100 * t / t.sum(), color=[colores.get(k, "gray") for k in t.index], edgecolor="black")
    ax.set_title(titulo); ax.set_ylabel("%"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

En el censo por volumen dominan las M; entre las estrellas brillantes del cielo, apenas hay M (y las que hay son gigantes, no enanas), y abundan las B y A, que son rarísimas en el vecindario.

## 5. Para ir más lejos: el sesgo de Malmquist, simulado

Simulamos estrellas de un mismo tipo, con magnitudes absolutas en campana ($M_0 = 1$, $\sigma = 0{,}5$), repartidas uniformemente en una esfera de 2000 pc. Nos quedamos con las que se ven más brillantes que $m = 9$ y calculamos su magnitud absoluta media.

In [ ]:
rng = np.random.default_rng(39)
n = 2_000_000
d = 2000 * rng.random(n) ** (1 / 3)                 # distancia con densidad uniforme en volumen
M0, sigma = 1.0, 0.5
Mabs = rng.normal(M0, sigma, n)
m = Mabs + 5 * np.log10(d / 10)
vistas = m < 9
print(f"media de M de todas:          {Mabs.mean():.3f}")
print(f"media de M de las que se ven: {Mabs[vistas].mean():.3f}")
print(f"predicción de Malmquist:      {M0 - 1.382 * sigma**2:.3f}")

## 6. Opcional: el vecindario según Gaia (sin probar)

Una consulta al archivo de Gaia para las fuentes con paralaje mayor de 100 mas (a menos de 10 pc) y error relativo pequeño. **No se ha podido probar** al escribir el cuaderno. Gaia no ve bien las estrellas muy brillantes (como Sirio) ni separa siempre las dobles muy cerradas, y no detecta casi ninguna enana marrón fría: por eso el censo de 2021 combinó Gaia con otros catálogos.

In [ ]:
USAR_GAIA = False

if USAR_GAIA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroquery"])
    from astroquery.gaia import Gaia
    consulta = """
        SELECT source_id, parallax, parallax_error, phot_g_mean_mag, bp_rp
        FROM gaiadr3.gaia_source
        WHERE parallax > 100 AND parallax_over_error > 10
    """
    g = Gaia.launch_job_async(consulta).get_results().to_pandas()
    g["M_G"] = g.phot_g_mean_mag + 5 * np.log10(g.parallax / 100)
    print(f"{len(g)} fuentes a menos de 10 pc")
    plt.scatter(g.bp_rp, g.M_G, s=6); plt.gca().invert_yaxis()
    plt.xlabel("color BP − RP"); plt.ylabel("magnitud absoluta G"); plt.show()
else:
    print("Consulta a Gaia desactivada. Cambia USAR_GAIA a True para probarla.")

## 7. Experimenta

1. Repite la tabla de tipos para las estrellas entre 10 y 25 pc. ¿Cambia la proporción? ¿Por qué?
2. Haz un diagrama de caja (`plt.boxplot`) de la magnitud absoluta de cada tipo espectral a menos de 25 pc.
3. En la simulación de Malmquist, prueba con $\sigma = 1$. ¿Se cumple todavía la fórmula?

In [ ]:
# Tu experimento aquí